<a href="https://colab.research.google.com/github/aneelabashir786/Pytorch/blob/main/Pytorch_NN_Module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import torch
import torch.nn as nn

class Model1(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(num_features,3),
        nn.ReLU(),
        nn.Linear(3,1),
        nn.Sigmoid()
    )


  def forward(self,x):
    out = self.network(x)
    return out

In [29]:
# import torch
# import torch.nn as nn

# class Model1(nn.Module):
#   def __init__(self,num_features):
#     super().__init__()
#     self.linear1 = nn.Linear(num_features,3)
#     self.relu=nn.ReLU()
#     self.linear2 = nn.Linear(3,1)
#     self.sigmoid = nn.Sigmoid()

#   def forward(self,x):
#     out = self.linear1(x)
#     out = self.relu(out)
#     out = self.linear2(out)
#     out = self.sigmoid(out)
#     return out

In [30]:
# import torch
# import torch.nn as nn

# class Model(nn.Module):
#   def __init__(self, num_features):
#     super().__init__()
#     self.linear = nn.Linear(num_features,1)
#     self.sigmoid = nn.Sigmoid()

#   def forward(self,x):
#     out = self.linear(x)
#     out = self.sigmoid(out)
#     return out


In [31]:
features = torch.rand(10,5)
features

tensor([[0.3156, 0.1366, 0.7594, 0.3626, 0.5424],
        [0.4087, 0.8165, 0.5179, 0.9458, 0.3361],
        [0.7694, 0.2555, 0.6931, 0.3859, 0.7020],
        [0.8073, 0.9473, 0.3148, 0.9353, 0.7864],
        [0.0449, 0.2353, 0.0321, 0.0379, 0.6222],
        [0.8103, 0.7063, 0.5188, 0.8028, 0.1963],
        [0.5636, 0.4335, 0.1441, 0.0751, 0.5590],
        [0.9172, 0.2319, 0.9118, 0.9071, 0.8626],
        [0.2856, 0.6342, 0.1187, 0.8185, 0.8523],
        [0.5755, 0.9369, 0.8424, 0.2736, 0.3617]])

In [32]:
model = Model1(features.shape[1])
model(features)

tensor([[0.4957],
        [0.4748],
        [0.4892],
        [0.4621],
        [0.4765],
        [0.4808],
        [0.4672],
        [0.4976],
        [0.4731],
        [0.4882]], grad_fn=<SigmoidBackward0>)

In [37]:
model.linear1.weight

AttributeError: 'Model1' object has no attribute 'linear1'

In [36]:
model.linear1.bias

AttributeError: 'Sequential' object has no attribute 'bias'

In [26]:
# To visualize

!pip install torchinfo

In [27]:
from torchinfo import summary
summary(model, input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model1                                   [10, 1]                   --
├─Linear: 1-1                            [10, 3]                   18
├─ReLU: 1-2                              [10, 3]                   --
├─Linear: 1-3                            [10, 1]                   4
├─Sigmoid: 1-4                           [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

# Improved

In [84]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , LabelEncoder

import torch
import torch.nn as nn

In [85]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [86]:
df.drop(columns=['id', 'Unnamed: 32'] , inplace= True)

In [87]:
df.shape

(569, 31)

In [88]:
X=df.drop(columns=['diagnosis'])
Y=df['diagnosis']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)

In [89]:
#Scalling

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [90]:
encoder = LabelEncoder()
Y_train = encoder.fit_transform(Y_train)
Y_test = encoder.transform(Y_test)

In [91]:
# Numpy array to tensor

X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
Y_train_tensor = torch.from_numpy(Y_train.astype(np.float32))
Y_test_tensor = torch.from_numpy(Y_test.astype(np.float32))

In [92]:
X_train_tensor.shape

torch.Size([455, 30])

## Model Defining

In [97]:
import torch
import torch.nn as nn

class MyModel(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.linear = nn.Linear(num_features,1)
    self.sigmoid = nn.Sigmoid()

  def forward(self,x):
    out = self.linear(x)
    out = self.sigmoid(out)
    return out

  # def loss_func(self, y_pred, y):
  #  # Clamp predictions to avoid log(0)
  #   epsilon = 1e-7
  #   y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

  #   # Calculate loss
  #   loss = -(Y_train_tensor * torch.log(y_pred) + (1 - Y_train_tensor) * torch.log(1 - y_pred)).mean()
  #   return loss


In [98]:
lr = 0.001
epochs = 25

loss_func = nn.BCELoss()

In [99]:

input_features = X_train_tensor.shape[1]
model3 = MyModel(input_features)

optimizor = torch.optim.SGD(model3.parameters(), lr=lr)

for epoch in range(epochs):
  y_pred = model3(X_train_tensor)
  loss = loss_func(y_pred,Y_train_tensor.reshape(-1,1))
  optimizor.zero_grad()
  loss.backward()
  optimizor.step()

  # with torch.no_grad():
  #   model3.linear.weight -= lr * model3.linear.weight.grad
  #   model3.linear.bias -= lr * model3.linear.bias.grad

  # model3.linear.weight.grad.zero_()
  # model3.linear.bias.grad.zero_()

  print(f'Epoch: {epoch+1} Loss: {loss.item()}')

Epoch: 1 Loss: 0.6257601380348206
Epoch: 2 Loss: 0.6240686178207397
Epoch: 3 Loss: 0.6223870515823364
Epoch: 4 Loss: 0.6207153797149658
Epoch: 5 Loss: 0.6190534234046936
Epoch: 6 Loss: 0.6174011826515198
Epoch: 7 Loss: 0.6157585978507996
Epoch: 8 Loss: 0.6141256093978882
Epoch: 9 Loss: 0.6125021576881409
Epoch: 10 Loss: 0.6108881235122681
Epoch: 11 Loss: 0.6092835068702698
Epoch: 12 Loss: 0.6076881289482117
Epoch: 13 Loss: 0.6061021685600281
Epoch: 14 Loss: 0.6045253276824951
Epoch: 15 Loss: 0.602957546710968
Epoch: 16 Loss: 0.6013988256454468
Epoch: 17 Loss: 0.5998492240905762
Epoch: 18 Loss: 0.5983084440231323
Epoch: 19 Loss: 0.5967766046524048
Epoch: 20 Loss: 0.5952535271644592
Epoch: 21 Loss: 0.5937392711639404
Epoch: 22 Loss: 0.5922336578369141
Epoch: 23 Loss: 0.5907366871833801
Epoch: 24 Loss: 0.5892482399940491
Epoch: 25 Loss: 0.5877682566642761


torch.optim is a sub-package within the PyTorch library that implements various optimization algorithms for training neural networks. It is used to minimize the loss function by adjusting the weights and biases of a model based on the computed gradients.

Key features and components of torch.optim include:
1. Built-in Optimizers

torch.optim provides implementations of common optimization algorithms, including:

SGD (Stochastic Gradient Descent): Implements SGD (optionally with momentum).

Adam: A popular adaptive learning rate optimization algorithm.

RMSprop: Root Mean Square Propagation, often used for recurrent neural networks.

Adagrad/Adadelta: Other adaptive algorithms.

L-BFGS: A quasi-Newton method.

model.parameters()

This method, part of the base torch.nn.Module class, returns an iterator over the Parameter objects associated with the model's layers and submodules.

Model Parameters:

These are internal variables within the model that are learned from the data during training. They determine how the model transforms input data into outputs (e.g., the weights and biases in a linear or convolutional layer).

torch.nn.Parameter:

Parameters are special subclasses of torch.Tensor that have a property which tells the PyTorch framework they require a gradient and should be updated during the optimization process.

Role in Optimization:

The list of parameters returned by model.parameters() is exactly what is passed to the torch.optim optimizer so that the optimizer knows which tensors need to be adjusted with each training step